# Twila Habab - C20361521
## TU059/FT ASD
### Programming for Big Data PROG9813

#### DONE IN DOCKER CONTAINER

In [1]:
# Links
# Kaggle Competition: https://www.kaggle.com/competitions/playground-series-s4e6/data
# UCI: https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success
# DOI: 10.24432/C5MC89

In [2]:
# Curl the datasets from Kaggle from my GitHub, no need for Kaggle APIs

In [3]:
%mkdir datasets

mkdir: cannot create directory ‘datasets’: File exists


In [4]:
!curl -sSL https://raw.githubusercontent.com/Hyper-TH/datasets_for_pbd_ca/refs/heads/master/train.csv -o "datasets/train.csv"

In [5]:
!curl -sSL https://raw.githubusercontent.com/Hyper-TH/datasets_for_pbd_ca/refs/heads/master/test.csv -o "datasets/test.csv"

In [6]:
# Import the necessary pyspark functions
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.ml.feature import StringIndexer, OneHotEncoder, IndexToString
from pyspark.ml import Pipeline
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

In [7]:
# spark.stop()

In [8]:
# Set up Session
from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .appName('C20361521 - Assignment 1') \
        .master('local[*]') \
        .getOrCreate()

In [9]:
# Load dataset into DFs
train_file_path = 'datasets/train.csv'

dataset = spark.read.format("csv").option("header", "true")\
    .option("inferSchema", "true")\
    .load(train_file_path)

train_df, test_df = dataset.randomSplit([0.7,0.3], seed=42)

In [10]:
# Cache it with a count operation
print('Training set size = ', train_df.cache().count())
print('Testing set size = ', test_df.cache().count())

# train_df.show(5)
# test_df.show(5)

Training set size =  53814
Testing set size =  22704


In [11]:
# Check schema
train_df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- Marital status: integer (nullable = true)
 |-- Application mode: integer (nullable = true)
 |-- Application order: integer (nullable = true)
 |-- Course: integer (nullable = true)
 |-- Daytime/evening attendance: integer (nullable = true)
 |-- Previous qualification: integer (nullable = true)
 |-- Previous qualification (grade): double (nullable = true)
 |-- Nacionality: integer (nullable = true)
 |-- Mother's qualification: integer (nullable = true)
 |-- Father's qualification: integer (nullable = true)
 |-- Mother's occupation: integer (nullable = true)
 |-- Father's occupation: integer (nullable = true)
 |-- Admission grade: double (nullable = true)
 |-- Displaced: integer (nullable = true)
 |-- Educational special needs: integer (nullable = true)
 |-- Debtor: integer (nullable = true)
 |-- Tuition fees up to date: integer (nullable = true)
 |-- Gender: integer (nullable = true)
 |-- Scholarship holder: integer (nullable = true)
 |-- Age 

In [12]:
test_df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- Marital status: integer (nullable = true)
 |-- Application mode: integer (nullable = true)
 |-- Application order: integer (nullable = true)
 |-- Course: integer (nullable = true)
 |-- Daytime/evening attendance: integer (nullable = true)
 |-- Previous qualification: integer (nullable = true)
 |-- Previous qualification (grade): double (nullable = true)
 |-- Nacionality: integer (nullable = true)
 |-- Mother's qualification: integer (nullable = true)
 |-- Father's qualification: integer (nullable = true)
 |-- Mother's occupation: integer (nullable = true)
 |-- Father's occupation: integer (nullable = true)
 |-- Admission grade: double (nullable = true)
 |-- Displaced: integer (nullable = true)
 |-- Educational special needs: integer (nullable = true)
 |-- Debtor: integer (nullable = true)
 |-- Tuition fees up to date: integer (nullable = true)
 |-- Gender: integer (nullable = true)
 |-- Scholarship holder: integer (nullable = true)
 |-- Age 

In [13]:
## Sanity cleaning: rename columns for easier identification

train_df = train_df\
    .withColumnRenamed("Marital Status", "Marital_Status")\
    .withColumnRenamed("Application mode", "Application_Mode")\
    .withColumnRenamed("Application order", "Application_Order")\
    .withColumnRenamed("Daytime/evening attendance", "DayNight_Attendance")\
    .withColumnRenamed("Previous qualification", "Prev_Qualification")\
    .withColumnRenamed("Previous qualification (grade)", "Prev_Qualification_Grade")\
    .withColumnRenamed("Nacionality", "Nationality")\
    .withColumnRenamed("Mother's Qualification", "Mother_Qualification")\
    .withColumnRenamed("Father's Qualification", "Father_Qualification")\
    .withColumnRenamed("Mother's occupation", "Mother_Occupation")\
    .withColumnRenamed("Father's occupation", "Father_Occupation")\
    .withColumnRenamed("Admission grade", "Admission_Grade")\
    .withColumnRenamed("Educational special needs", "Special_Education")\
    .withColumnRenamed("Tuition fees up to date", "Up_To_Date_Tuition")\
    .withColumnRenamed("Scholarship holder", "Scholarship")\
    .withColumnRenamed("Age at enrollment", "Enrollment_Age")\
    .withColumnRenamed("Curricular units 1st sem (credited)", "1st_Sem_units_Credited")\
    .withColumnRenamed("Curricular units 1st sem (enrolled)", "1st_Sem_units_Enrolled")\
    .withColumnRenamed("Curricular units 1st sem (evaluations)", "1st_Sem_units_Evaluations")\
    .withColumnRenamed("Curricular units 1st sem (approved)", "1st_Sem_units_Approved")\
    .withColumnRenamed("Curricular units 1st sem (grade)", "1st_Sem_units_Grade")\
    .withColumnRenamed("Curricular units 1st sem (without evaluations)", "1st_Sem_units_NoEval")\
    .withColumnRenamed("Curricular units 2nd sem (credited)", "2nd_Sem_units_Credited")\
    .withColumnRenamed("Curricular units 2nd sem (enrolled)", "2nd_Sem_units_Enrolled")\
    .withColumnRenamed("Curricular units 2nd sem (evaluations)", "2nd_Sem_units_Evaluations")\
    .withColumnRenamed("Curricular units 2nd sem (approved)", "2nd_Sem_units_Approved")\
    .withColumnRenamed("Curricular units 2nd sem (grade)", "2nd_Sem_units_Grade")\
    .withColumnRenamed("Curricular units 2nd sem (without evaluations)", "2nd_Sem_units_NoEval")\
    .withColumnRenamed("Unemployment rate", "Unemployment_Rate")\
    .withColumnRenamed("Inflation rate", "Inflation_Rate")


# Do the same for test_df
test_df= test_df\
    .withColumnRenamed("Marital Status", "Marital_Status")\
    .withColumnRenamed("Application mode", "Application_Mode")\
    .withColumnRenamed("Application order", "Application_Order")\
    .withColumnRenamed("Daytime/evening attendance", "DayNight_Attendance")\
    .withColumnRenamed("Previous qualification", "Prev_Qualification")\
    .withColumnRenamed("Previous qualification (grade)", "Prev_Qualification_Grade")\
    .withColumnRenamed("Nacionality", "Nationality")\
    .withColumnRenamed("Mother's Qualification", "Mother_Qualification")\
    .withColumnRenamed("Father's Qualification", "Father_Qualification")\
    .withColumnRenamed("Mother's occupation", "Mother_Occupation")\
    .withColumnRenamed("Father's occupation", "Father_Occupation")\
    .withColumnRenamed("Admission grade", "Admission_Grade")\
    .withColumnRenamed("Educational special needs", "Special_Education")\
    .withColumnRenamed("Tuition fees up to date", "Up_To_Date_Tuition")\
    .withColumnRenamed("Scholarship holder", "Scholarship")\
    .withColumnRenamed("Age at enrollment", "Enrollment_Age")\
    .withColumnRenamed("Curricular units 1st sem (credited)", "1st_Sem_units_Credited")\
    .withColumnRenamed("Curricular units 1st sem (enrolled)", "1st_Sem_units_Enrolled")\
    .withColumnRenamed("Curricular units 1st sem (evaluations)", "1st_Sem_units_Evaluations")\
    .withColumnRenamed("Curricular units 1st sem (approved)", "1st_Sem_units_Approved")\
    .withColumnRenamed("Curricular units 1st sem (grade)", "1st_Sem_units_Grade")\
    .withColumnRenamed("Curricular units 1st sem (without evaluations)", "1st_Sem_units_NoEval")\
    .withColumnRenamed("Curricular units 2nd sem (credited)", "2nd_Sem_units_Credited")\
    .withColumnRenamed("Curricular units 2nd sem (enrolled)", "2nd_Sem_units_Enrolled")\
    .withColumnRenamed("Curricular units 2nd sem (evaluations)", "2nd_Sem_units_Evaluations")\
    .withColumnRenamed("Curricular units 2nd sem (approved)", "2nd_Sem_units_Approved")\
    .withColumnRenamed("Curricular units 2nd sem (grade)", "2nd_Sem_units_Grade")\
    .withColumnRenamed("Curricular units 2nd sem (without evaluations)", "2nd_Sem_units_NoEval")\
    .withColumnRenamed("Unemployment rate", "Unemployment_Rate")\
    .withColumnRenamed("Inflation rate", "Inflation_Rate")

# train_df.show(5)    
# test_df.show(5)

## Data Exploration

In [14]:
# Let's explore the different demographics of the datasets
train_df.select("Nationality").where(col("Nationality").isNotNull()) \
       .groupBy("Nationality") \
       .count() \
       .orderBy("count", ascending=False) \
       .show(n=train_df.count(), truncate=False)

test_df.select("Nationality").where(col("Nationality").isNotNull()) \
       .groupBy("Nationality") \
       .count() \
       .orderBy("count", ascending=False) \
       .show(n=train_df.count(), truncate=False)

# 1 - Portuguese; 2 - German; 6 - Spanish; 11 - Italian; 13 - Dutch; 14 - English; 
# 17 - Lithuanian; 21 - Angolan; 22 - Cape Verdean; 24 - Guinean; 25 - Mozambican; 26 - Santomean; 
# 32 - Turkish; 41 - Brazilian; 62 - Romanian; 100 - Moldova (Republic of); 101 - Mexican; 
# 103 - Ukrainian; 105 - Russian; 108 - Cuban; 109 - Colombian

+-----------+-----+
|Nationality|count|
+-----------+-----+
|1          |53443|
|41         |160  |
|26         |48   |
|6          |42   |
|22         |40   |
|2          |12   |
|24         |11   |
|11         |11   |
|100        |9    |
|101        |8    |
|103        |7    |
|105        |7    |
|25         |6    |
|21         |5    |
|62         |3    |
|17         |1    |
|32         |1    |
+-----------+-----+

+-----------+-----+
|Nationality|count|
+-----------+-----+
|1          |22570|
|41         |61   |
|26         |19   |
|22         |16   |
|6          |14   |
|103        |5    |
|24         |4    |
|11         |4    |
|62         |3    |
|109        |2    |
|105        |2    |
|17         |1    |
|21         |1    |
|101        |1    |
|2          |1    |
+-----------+-----+



In [15]:
# Check the summary of the Age at enrollment column
train_df.select("Enrollment_Age").summary().show()
test_df.select("Enrollment_Age").summary().show()

## Same quarters, min and max

+-------+-----------------+
|summary|   Enrollment_Age|
+-------+-----------------+
|  count|            53814|
|   mean| 22.2837179916007|
| stddev|6.893586283991445|
|    min|               17|
|    25%|               18|
|    50%|               19|
|    75%|               23|
|    max|               70|
+-------+-----------------+

+-------+------------------+
|summary|    Enrollment_Age|
+-------+------------------+
|  count|             22704|
|   mean|22.266649048625794|
| stddev|  6.87906785361406|
|    min|                17|
|    25%|                18|
|    50%|                19|
|    75%|                23|
|    max|                70|
+-------+------------------+



In [16]:
# AGE GROUP COLUMN
# Since we know the min max of ages, we can group them
# min = 17; max = 70
# 17-24; 25-34; 35-44; 45-54; 55-64; 65-70;

# Creating a new column and inserting values with nested whens
train_df = train_df.withColumn("Age_Group",
    when(col("Enrollment_Age") <= 24, "17-24")\
    .when(col("Enrollment_Age") <= 34, "25-34")\
    .when(col("Enrollment_Age") <= 44, "35-44")\
    .when(col("Enrollment_Age") <= 54, "45-54")\
    .when(col("Enrollment_Age") <= 64, "55-64")\
    .when(col("Enrollment_Age") <= 70, "65-70")
)

# Do the same for test_df
test_df = test_df.withColumn("Age_Group",
    when(col("Enrollment_Age") <= 24, "17-24")\
    .when(col("Enrollment_Age") <= 34, "25-34")\
    .when(col("Enrollment_Age") <= 44, "35-44")\
    .when(col("Enrollment_Age") <= 54, "45-54")\
    .when(col("Enrollment_Age") <= 64, "55-64")\
    .when(col("Enrollment_Age") <= 70, "65-70")
)

train_df.select("Age_Group").distinct().show(n=train_df.count())

+---------+
|Age_Group|
+---------+
|    45-54|
|    55-64|
|    17-24|
|    35-44|
|    25-34|
|    65-70|
+---------+



In [17]:
train_df.select("Age_Group")\
    .groupBy("Age_Group")\
    .agg(count("Age_Group").alias("Count"))\
    .orderBy("Count", ascending=False)\
    .show()

test_df.select("Age_Group")\
    .groupBy("Age_Group")\
    .agg(count("Age_Group").alias("Count"))\
    .orderBy("Count", ascending=False)\
    .show()

+---------+-----+
|Age_Group|Count|
+---------+-----+
|    17-24|42240|
|    25-34| 7535|
|    35-44| 2774|
|    45-54| 1123|
|    55-64|  124|
|    65-70|   18|
+---------+-----+

+---------+-----+
|Age_Group|Count|
+---------+-----+
|    17-24|17921|
|    25-34| 3068|
|    35-44| 1153|
|    45-54|  509|
|    55-64|   51|
|    65-70|    2|
+---------+-----+



In [18]:
# Distribution of the Admission Grade
train_df.select("Admission_Grade").summary().show()

+-------+------------------+
|summary|   Admission_Grade|
+-------+------------------+
|  count|             53814|
|   mean|125.39263367803787|
| stddev|  12.5955755773937|
|    min|              95.0|
|    25%|             118.0|
|    50%|             124.7|
|    75%|             132.0|
|    max|             190.0|
+-------+------------------+



In [19]:
# Distribution of Unemployment Rate
train_df.select("Unemployment_Rate").summary().show()

+-------+------------------+
|summary| Unemployment_Rate|
+-------+------------------+
|  count|             53814|
|   mean|11.521260266844916|
| stddev| 2.655343651685822|
|    min|               7.6|
|    25%|               9.4|
|    50%|              11.1|
|    75%|              12.7|
|    max|              16.2|
+-------+------------------+



In [20]:
# Scholarship status
train_df.select(when(col("Scholarship") == 0, "No").otherwise("Yes").alias("Scholarship_Status")).groupBy("Scholarship_Status").count().show()

+------------------+-----+
|Scholarship_Status|count|
+------------------+-----+
|                No|40591|
|               Yes|13223|
+------------------+-----+



In [21]:
# Will use TempView to explore and analyse the dataframes with SQL
train_df.createOrReplaceTempView("train_tab")
test_df.createOrReplaceTempView("test_tab")

In [22]:
# What is the most common qualification?
spark.sql(
    """SELECT `Prev_Qualification`, count(*) AS COUNT FROM train_tab 
        GROUP BY `Prev_Qualification` ORDER BY COUNT DESC"""
).show(train_df.count())

+------------------+-----+
|Prev_Qualification|COUNT|
+------------------+-----+
|                 1|47313|
|                19| 2049|
|                39| 2033|
|                 3|  971|
|                12|  649|
|                 9|  206|
|                40|  184|
|                42|  167|
|                 6|   71|
|                 2|   61|
|                10|   28|
|                38|   27|
|                43|   25|
|                 4|   13|
|                15|    6|
|                 5|    3|
|                37|    3|
|                14|    2|
|                17|    2|
|                36|    1|
+------------------+-----+



In [23]:
# Find the most common finished education by age_group
spark.sql("""
    SELECT `Prev_Qualification` as Most_Common_Qualification, `Age_Group` AS Age_Group
        FROM (
            SELECT `Prev_Qualification`, `Age_Group`,
                RANK() OVER (PARTITION BY `Age_Group` ORDER BY cnt DESC) as rnk 
            FROM (
                SELECT `Prev_Qualification`, `Age_Group`, COUNT(*) AS cnt
                FROM train_tab 
                GROUP BY `Prev_Qualification`, `Age_Group`
            ) AS qual_count
        ) AS qual_rank_per_age
        WHERE rnk = 1 --get the top per group
    """).show(train_df.count())

+-------------------------+---------+
|Most_Common_Qualification|Age_Group|
+-------------------------+---------+
|                        1|    17-24|
|                        1|    25-34|
|                        1|    35-44|
|                        1|    45-54|
|                       12|    55-64|
|                       19|    65-70|
+-------------------------+---------+



In [24]:
# Find the most common finished education by Enrollment_Age
spark.sql("""
    SELECT `Prev_Qualification` as Most_Common_Qualification, `Enrollment_Age` AS Age
        FROM (
            SELECT `Prev_Qualification`, `Enrollment_Age`,
                RANK() OVER (PARTITION BY `Enrollment_Age` ORDER BY cnt DESC) as rnk 
            FROM (
                SELECT `Prev_Qualification`, `Enrollment_Age`, COUNT(*) AS cnt
                FROM train_tab 
                GROUP BY `Prev_Qualification`, `Enrollment_Age`
            ) AS qual_count
        ) AS qual_rank_per_age
        WHERE rnk = 1
    """).show(train_df.count())

+-------------------------+---+
|Most_Common_Qualification|Age|
+-------------------------+---+
|                        1| 17|
|                        1| 18|
|                        1| 19|
|                        1| 20|
|                        1| 21|
|                        1| 22|
|                        1| 23|
|                        1| 24|
|                        1| 25|
|                        1| 26|
|                        1| 27|
|                        1| 28|
|                        1| 29|
|                        1| 30|
|                        1| 31|
|                        1| 32|
|                        1| 33|
|                        1| 34|
|                        1| 35|
|                        1| 36|
|                        1| 37|
|                        1| 38|
|                        1| 39|
|                        1| 40|
|                        1| 41|
|                        1| 42|
|                        1| 43|
|                        1| 44|
|       

In [25]:
# Find the most common coures per gender
spark.sql("""
    SELECT `Course` as Most_Common_Course, `Gender`
        FROM (
            SELECT `Course`, `Gender`,
                RANK() OVER (PARTITION BY `Gender` ORDER BY cnt DESC) as rnk 
            FROM (
                SELECT `Course`, `Gender`, COUNT(*) AS cnt
                FROM train_tab 
                GROUP BY `Course`, `Gender`
            ) AS course_count
        ) AS course_rank_per_age
        WHERE rnk = 1
    """).show(train_df.count())

# 0 (Female) => 9500 Nursing
# 1 (Male) => 9147 Management 

+------------------+------+
|Most_Common_Course|Gender|
+------------------+------+
|              9500|     0|
|              9147|     1|
+------------------+------+



## Data Cleaning and Preparation

In [26]:
# Although the competition and the original source have stated
# that the datasets do not contain any NULL values, it is still
# good practice to check for any of it.

In [27]:
# Check any null values
train_df.select([count(when(col(c).isNull(), c)).alias(c) for c in train_df.columns]).show() 
test_df.select([count(when(col(c).isNull(), c)).alias(c) for c in test_df.columns]).show()

+---+--------------+----------------+-----------------+------+-------------------+------------------+------------------------+-----------+--------------------+--------------------+-----------------+-----------------+---------------+---------+-----------------+------+------------------+------+-----------+--------------+-------------+----------------------+----------------------+-------------------------+----------------------+-------------------+--------------------+----------------------+----------------------+-------------------------+----------------------+-------------------+--------------------+-----------------+--------------+---+------+---------+
| id|Marital_Status|Application_Mode|Application_Order|Course|DayNight_Attendance|Prev_Qualification|Prev_Qualification_Grade|Nationality|Mother_Qualification|Father_Qualification|Mother_Occupation|Father_Occupation|Admission_Grade|Displaced|Special_Education|Debtor|Up_To_Date_Tuition|Gender|Scholarship|Enrollment_Age|International|1st_S

## Feature Engineering

## Build a model that predicts the academic risk of students in higher education

### Convert Categorical Variables

In [28]:
# Cast the categorical columns into string type so that it can be passed onto the string indexer
train_df = train_df\
    .withColumn("Marital_Status", col("Marital_Status").cast("string"))\
    .withColumn("Application_Mode", col("Application_Mode").cast("string"))\
    .withColumn("Application_Order", col("Application_Order").cast("string"))\
    .withColumn("Course", col("Course").cast("string"))\
    .withColumn("Prev_Qualification", col("Prev_Qualification").cast("string"))\
    .withColumn("Nationality", col("Nationality").cast("string"))\
    .withColumn("Mother_Qualification", col("Mother_Qualification").cast("string"))\
    .withColumn("Father_Qualification", col("Father_Qualification").cast("string"))\
    .withColumn("Mother_Occupation", col("Mother_Occupation").cast("string"))\
    .withColumn("Father_Occupation", col("Father_Occupation").cast("string"))\
    .withColumn("Age_Group", col("Age_Group").cast("string"))

test_df = test_df\
    .withColumn("Marital_Status", col("Marital_Status").cast("string"))\
    .withColumn("Application_Mode", col("Application_Mode").cast("string"))\
    .withColumn("Application_Order", col("Application_Order").cast("string"))\
    .withColumn("Course", col("Course").cast("string"))\
    .withColumn("Prev_Qualification", col("Prev_Qualification").cast("string"))\
    .withColumn("Nationality", col("Nationality").cast("string"))\
    .withColumn("Mother_Qualification", col("Mother_Qualification").cast("string"))\
    .withColumn("Father_Qualification", col("Father_Qualification").cast("string"))\
    .withColumn("Mother_Occupation", col("Mother_Occupation").cast("string"))\
    .withColumn("Father_Occupation", col("Father_Occupation").cast("string"))\
    .withColumn("Age_Group", col("Age_Group").cast("string"))

### Feature Engineering

In [29]:
# StringIndexer and OneHotEncoder

# Categorical Columns
categoricalCols = ["Marital_Status","Application_Mode","Application_Order",
                   "Course","Prev_Qualification","Nationality",
                   "Mother_Qualification", "Father_Qualification","Mother_Occupation",
                  "Father_Occupation"]

# Estimators, return functions that will later use for transforming the dataset.
# ESTIMATOR
stringIndexer = StringIndexer(inputCols=categoricalCols, outputCols=[x + "Index" for x in categoricalCols], handleInvalid="keep")
encoder = OneHotEncoder(inputCols=stringIndexer.getOutputCols(), outputCols=[x + "OHE" for x in categoricalCols])

# The label 'Target' is also a string value, convert it to numeric
labelToIndex = StringIndexer(inputCol="Target", outputCol="label")

In [30]:
# TRANSFORMER
stringIndexerModel = stringIndexer.fit(test_df)
stringIndexerModel.transform(test_df).select("Marital_Status", "Target").show(5, truncate=False)

+--------------+--------+
|Marital_Status|Target  |
+--------------+--------+
|1             |Dropout |
|1             |Graduate|
|1             |Dropout |
|1             |Graduate|
|2             |Dropout |
+--------------+--------+
only showing top 5 rows


In [31]:
# FEATURE VECTOR
numericCols = ["DayNight_Attendance", "Prev_Qualification_Grade", "Admission_Grade",
              "Displaced", "Special_Education", "Debtor",  "Up_To_Date_Tuition", 
              "Gender", "Scholarship", "Enrollment_Age", "International",
              "1st_Sem_units_Credited", "1st_Sem_units_Enrolled", "1st_Sem_units_Evaluations",
              "1st_Sem_units_Approved", "1st_Sem_units_Grade", "1st_Sem_units_NoEval",
              "2nd_Sem_units_Credited", "2nd_Sem_units_Enrolled", "2nd_Sem_units_Evaluations",
              "2nd_Sem_units_Approved", "2nd_Sem_units_Grade", "2nd_Sem_units_NoEval",
              "Unemployment_Rate", "Inflation_Rate", "GDP"]


assemblerInputs =[c + "OHE" for c in categoricalCols] + numericCols
vecAssembler = VectorAssembler(inputCols=assemblerInputs, outputCol="features")

### Models

In [64]:
# Define the models
lr = LogisticRegression(featuresCol="features", labelCol="label", regParam=1.0)
dtg = DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=10, impurity="Gini")
dte = DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=10, impurity="Entropy")
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=3, maxDepth=2,  seed=42)

In [65]:
# Define the pipeline based on stages created in prev steps
lr_pipeline = Pipeline(stages=[stringIndexer, encoder, labelToIndex, vecAssembler, lr])
dtg_pipeline = Pipeline(stages=[stringIndexer, encoder, labelToIndex, vecAssembler, dtg])
dte_pipeline = Pipeline(stages=[stringIndexer, encoder, labelToIndex, vecAssembler, dte])
rf_pipeline = Pipeline(stages=[stringIndexer, encoder, labelToIndex, vecAssembler, rf])

# Define the pipeline model
pipelineModel_lr = lr_pipeline.fit(train_df)
pipelineModel_dtg = dtg_pipeline.fit(train_df)
pipelineModel_dte = dte_pipeline.fit(train_df)
pipelineModel_rf = rf_pipeline.fit(train_df)

# Apply the pipeline model to the test dataset
predDF_lr = pipelineModel_lr.transform(test_df)
predDF_dtg = pipelineModel_dtg.transform(test_df)
predDF_dte = pipelineModel_dte.transform(test_df)
predDF_rf = pipelineModel_rf.transform(test_df)

In [66]:
# Evaluate the models
mcEvaluator = MulticlassClassificationEvaluator(metricName="accuracy")

print(f"Accuracy of Logistic Regression Model: {mcEvaluator.evaluate(predDF_lr)}")
print(f"Accuracy of Decision Tree Classifier (Gini) Model: {mcEvaluator.evaluate(predDF_dtg)}")
print(f"Accuracy of Decision Tree Classifier (Entropy) Model: {mcEvaluator.evaluate(predDF_dte)}")
print(f"Accuracy of Random Forest Classifier Model: {mcEvaluator.evaluate(predDF_rf)}")

Accuracy of Logistic Regression Model: 0.7324260042283298
Accuracy of Decision Tree Classifier (Gini) Model: 0.8127642706131079
Accuracy of Decision Tree Classifier (Entropy) Model: 0.8140856236786469
Accuracy of Random Forest Classifier Model: 0.7300035236081748


### Models Investigation

In [49]:
# SQL for additional analysis
# create TempView for SQL
predDF_lr.createOrReplaceTempView("Logistic_Regression_Predictions")
predDF_dt.createOrReplaceTempView("Decision_Tree_Predictions")
predDF_rf.createOrReplaceTempView("Forest_Tree_Predictions")

sql = """select id, Target, prediction from Logistic_Regression_Predictions where prediction = 1"""
spark.sql(sql).show(n=25)

+---+--------+----------+
| id|  Target|prediction|
+---+--------+----------+
|  2| Dropout|       1.0|
|  8| Dropout|       1.0|
| 13| Dropout|       1.0|
| 14| Dropout|       1.0|
| 29|Graduate|       1.0|
| 39| Dropout|       1.0|
| 43| Dropout|       1.0|
| 45| Dropout|       1.0|
| 47| Dropout|       1.0|
| 49| Dropout|       1.0|
| 55|Enrolled|       1.0|
| 62| Dropout|       1.0|
|118| Dropout|       1.0|
|140| Dropout|       1.0|
|143| Dropout|       1.0|
|164| Dropout|       1.0|
|192| Dropout|       1.0|
|202| Dropout|       1.0|
|204| Dropout|       1.0|
|206| Dropout|       1.0|
|233| Dropout|       1.0|
|254| Dropout|       1.0|
|261| Dropout|       1.0|
|263| Dropout|       1.0|
|288| Dropout|       1.0|
+---+--------+----------+
only showing top 25 rows


### HyperParameter Tuning

In [68]:
lr_paramGrid = (ParamGridBuilder()
             .addGrid(lr.regParam, [0.1, 0.5, 2.0])
             .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
             .build())

dtg_paramGrid = (ParamGridBuilder()
             .addGrid(dt.minInstancesPerNode, [1, 10, 20])
             .addGrid(dt.maxDepth, [1, 10, 20])
             .build())

dte_paramGrid = (ParamGridBuilder()
             .addGrid(dt.minInstancesPerNode, [1, 10, 20])
             .addGrid(dt.maxDepth, [1, 10, 20])
             .build())

rf_paramGrid = (ParamGridBuilder()
            .addGrid(rf.numTrees, [20, 40, 50]) 
            .addGrid(rf.maxDepth, [5, 10, 15])      
            .build())

In [69]:
# Create a 3-fold CrossValidator for model 1
cv = CrossValidator(estimator=lr_pipeline, estimatorParamMaps=lr_paramGrid, evaluator=mcEvaluator, numFolds=3, parallelism=4)

# Run cross validations. This may take a while and returns the best model found 
cvModel_lr = cv.fit(train_df)

In [74]:
# Create a 3-fold CrossValidator for model 2
cv_1 = CrossValidator(estimator=dtg_pipeline, estimatorParamMaps=dtg_paramGrid, evaluator=mcEvaluator, numFolds=3, parallelism=4)

cvModel_dtg = cv_1.fit(train_df)

In [75]:
# Create a 3-fold CrossValidator for model 3
cv_2 = CrossValidator(estimator=dte_pipeline, estimatorParamMaps=dte_paramGrid, evaluator=mcEvaluator, numFolds=3, parallelism=4)

cvModel_dte = cv_1.fit(train_df)

In [72]:
# Create a 3-fold CrossValidator for model 4
cv_3 = CrossValidator(estimator=rf_pipeline, estimatorParamMaps=rf_paramGrid, evaluator=mcEvaluator, numFolds=3, parallelism=4)

cvModel_rf = cv_3.fit(train_df)

In [76]:
# Evaluate the models
cvPredDF_lr = cvModel_lr.transform(test_df)
cvPredDF_dtg = cvModel_dtg.transform(test_df)
cvPredDF_dte = cvModel_dte.transform(test_df)
cvPredDF_rf = cvModel_rf.transform(test_df)

print(f"Accuracy of Logistic Regression Model: {mcEvaluator.evaluate(predDF_lr)}")
print(f"Accuracy of Decision Tree Classifier (Gini) Model: {mcEvaluator.evaluate(predDF_dtg)}")
print(f"Accuracy of Decision Tree Classifier (Entropy) Model: {mcEvaluator.evaluate(cvPredDF_dte)}")
print(f"Accuracy of Random Forest Classifier Model: {mcEvaluator.evaluate(predDF_rf)}")
print("===AFTER HYPERTUNING===")
print(f"Accuracy of Logistic Regression Model: {mcEvaluator.evaluate(cvPredDF_lr)}")
print(f"Accuracy of Decision Tree Classifier (Gini) Model: {mcEvaluator.evaluate(cvPredDF_dtg)}")
print(f"Accuracy of Decision Tree Classifier (Entropy) Model: {mcEvaluator.evaluate(cvPredDF_dte)}")
print(f"Accuracy of Random Forest Classifier Model: {mcEvaluator.evaluate(cvPredDF_rf)}")


Accuracy of Logistic Regression Model: 0.7324260042283298
Accuracy of Decision Tree Classifier (Gini) Model: 0.8127642706131079
Accuracy of Decision Tree Classifier (Entropy) Model: 0.8127642706131079
Accuracy of Random Forest Classifier Model: 0.7300035236081748
===AFTER HYPERTUNING===
Accuracy of Logistic Regression Model: 0.8058491895701198
Accuracy of Decision Tree Classifier (Gini) Model: 0.8127642706131079
Accuracy of Decision Tree Classifier (Entropy) Model: 0.8127642706131079
Accuracy of Random Forest Classifier Model: 0.821000704721635


## Kaggle Submission

In [55]:
# Load the testing dataset provided by kaggle
test_file_path = 'datasets/test.csv'

testing_df = spark.read.format("csv").option("header", "true")\
    .option("inferSchema", "true")\
    .load(test_file_path)

In [56]:
# Cache it
print('Testing set size = ', testing_df.cache().count())

Testing set size =  51012


In [57]:
# Apply the changes done
testing_df = testing_df\
    .withColumnRenamed("Marital Status", "Marital_Status")\
    .withColumnRenamed("Application mode", "Application_Mode")\
    .withColumnRenamed("Application order", "Application_Order")\
    .withColumnRenamed("Daytime/evening attendance", "DayNight_Attendance")\
    .withColumnRenamed("Previous qualification", "Prev_Qualification")\
    .withColumnRenamed("Previous qualification (grade)", "Prev_Qualification_Grade")\
    .withColumnRenamed("Nacionality", "Nationality")\
    .withColumnRenamed("Mother's Qualification", "Mother_Qualification")\
    .withColumnRenamed("Father's Qualification", "Father_Qualification")\
    .withColumnRenamed("Mother's occupation", "Mother_Occupation")\
    .withColumnRenamed("Father's occupation", "Father_Occupation")\
    .withColumnRenamed("Admission grade", "Admission_Grade")\
    .withColumnRenamed("Educational special needs", "Special_Education")\
    .withColumnRenamed("Tuition fees up to date", "Up_To_Date_Tuition")\
    .withColumnRenamed("Scholarship holder", "Scholarship")\
    .withColumnRenamed("Age at enrollment", "Enrollment_Age")\
    .withColumnRenamed("Curricular units 1st sem (credited)", "1st_Sem_units_Credited")\
    .withColumnRenamed("Curricular units 1st sem (enrolled)", "1st_Sem_units_Enrolled")\
    .withColumnRenamed("Curricular units 1st sem (evaluations)", "1st_Sem_units_Evaluations")\
    .withColumnRenamed("Curricular units 1st sem (approved)", "1st_Sem_units_Approved")\
    .withColumnRenamed("Curricular units 1st sem (grade)", "1st_Sem_units_Grade")\
    .withColumnRenamed("Curricular units 1st sem (without evaluations)", "1st_Sem_units_NoEval")\
    .withColumnRenamed("Curricular units 2nd sem (credited)", "2nd_Sem_units_Credited")\
    .withColumnRenamed("Curricular units 2nd sem (enrolled)", "2nd_Sem_units_Enrolled")\
    .withColumnRenamed("Curricular units 2nd sem (evaluations)", "2nd_Sem_units_Evaluations")\
    .withColumnRenamed("Curricular units 2nd sem (approved)", "2nd_Sem_units_Approved")\
    .withColumnRenamed("Curricular units 2nd sem (grade)", "2nd_Sem_units_Grade")\
    .withColumnRenamed("Curricular units 2nd sem (without evaluations)", "2nd_Sem_units_NoEval")\
    .withColumnRenamed("Unemployment rate", "Unemployment_Rate")\
    .withColumnRenamed("Inflation rate", "Inflation_Rate")

testing_df = testing_df.withColumn("Age_Group",
    when(col("Enrollment_Age") <= 24, "17-24")\
    .when(col("Enrollment_Age") <= 34, "25-34")\
    .when(col("Enrollment_Age") <= 44, "35-44")\
    .when(col("Enrollment_Age") <= 54, "45-54")\
    .when(col("Enrollment_Age") <= 64, "55-64")\
    .when(col("Enrollment_Age") <= 70, "65-70")
)

testing_df = testing_df\
    .withColumn("Marital_Status", col("Marital_Status").cast("string"))\
    .withColumn("Application_Mode", col("Application_Mode").cast("string"))\
    .withColumn("Application_Order", col("Application_Order").cast("string"))\
    .withColumn("Course", col("Course").cast("string"))\
    .withColumn("Prev_Qualification", col("Prev_Qualification").cast("string"))\
    .withColumn("Nationality", col("Nationality").cast("string"))\
    .withColumn("Mother_Qualification", col("Mother_Qualification").cast("string"))\
    .withColumn("Father_Qualification", col("Father_Qualification").cast("string"))\
    .withColumn("Mother_Occupation", col("Mother_Occupation").cast("string"))\
    .withColumn("Father_Occupation", col("Father_Occupation").cast("string"))\
    .withColumn("Age_Group", col("Age_Group").cast("string"))

In [77]:
# Now we need to prepare the kaggle submission

# Since the testing data does not have the Target column, 
# we need to create a transformer where 'Target' exists
labelIndexerModel = StringIndexer(inputCol="Target", outputCol="label").fit(train_df)

# Now we run our ml model to our testing data to create the prediction column
# we'll use the highest accuracy model
predictions_df = cvModel_rf.transform(testing_df)

# We need to create a converter so that we could create a Target column
# based on the values of the predictions column
# and using the labelsArray from the transformer created earlier
converter = IndexToString(inputCol="prediction", outputCol="Target", labels=labelIndexerModel.labelsArray[0])

# Now we get the final df
final_df = converter.transform(predictions_df)

In [78]:
# We only need the id and Target columns
kaggle_submission = final_df.select("id", "Target")

In [79]:
# The competition is only expecting 51012 rows, so check it
kaggle_submission.count()

51012

In [80]:
kaggle_submission.show()

+-----+--------+
|   id|  Target|
+-----+--------+
|76518| Dropout|
|76519|Graduate|
|76520|Graduate|
|76521|Graduate|
|76522|Enrolled|
|76523|Graduate|
|76524|Graduate|
|76525|Graduate|
|76526| Dropout|
|76527|Graduate|
|76528|Graduate|
|76529|Graduate|
|76530|Graduate|
|76531|Enrolled|
|76532| Dropout|
|76533|Enrolled|
|76534|Graduate|
|76535|Graduate|
|76536|Enrolled|
|76537|Graduate|
+-----+--------+
only showing top 20 rows


In [81]:
# Now we convert to csv
# Use coalesce to avoid making multiple files in one folder
kaggle_submission.coalesce(1).write.csv('c20361521', header=True, mode="overwrite")